## Notebook43

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
! pip install umap-learn --quiet
! python -m spacy download en_core_web_sm
! python -m spacy download fr_core_news_sm

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c
import numpy as np

import funs
from funs import DSText, umap_text

import spacy

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
meta = pl.read_csv(ub + "data/wiki_uk_meta.csv.gz")
text_en = pl.read_csv(ub + "data/wiki_uk_authors_text.csv")
text_fr = pl.read_csv(ub + "data/wiki_uk_authors_text_fr.csv")

In [ ]:
nlp_en = spacy.load("en_core_web_sm")
nlp_fr = spacy.load("fr_core_news_sm")

### Questions

`meta` holds one row per author, with a birth and death year, a literary
`era`, a `gender`, and the `link`/`short` names used elsewhere in the corpus.
`text_en` has one row per author with the full text of their English
Wikipedia page; `text_fr` is the same idea for the French edition. This is
the same 75-author corpus the chapter itself builds its cross-language
section on, because it is the only French-paired text in the whole
catalogue — there is no second bilingual corpus to swap in instead. The
first question below investigates a real narrowing this notebook has to
make before doing anything else: `text_fr` does not have 75 rows. Print
results unless a question says otherwise.

1. How many of the 75 authors in `text_en` also have a row in `text_fr`?
Anti-join to find the author(s) missing a French page, and join the result
to `meta` for their era and gender.

In [ ]:
n_common = (
    text_en
    .join(text_fr.select(c.doc_id), on="doc_id")
    .height
)
print(n_common, "of", text_en.height, "authors have a French page")

missing = text_en.join(text_fr.select(c.doc_id), on="doc_id", how="anti")

(
    missing
    .join(meta, on="doc_id")
    .select(c.doc_id, c.era, c.gender)
)

Now rank all 75 authors by the character length of their English page, and
report where the two missing authors fall in that ranking.

In [ ]:
(
    text_en
    .with_columns(n_chars = c.text.str.len_chars())
    .with_columns(rank = c.n_chars.rank())
    .filter(c.doc_id.is_in(missing.select(c.doc_id).to_series()))
    .select(c.doc_id, c.n_chars, c.rank)
)

Only 73 of the 75 authors have a French Wikipedia page. Both authors missing
one, Rex Warner and Edward Upward, are Twentieth Century writers, but era
alone does not explain the gap: plenty of other Twentieth Century authors in
this corpus do have a French page. Page length does explain it. Rex Warner
has the single shortest English page in the entire 75-author corpus, and
Edward Upward has the third shortest. The two authors with no French
Wikipedia page at all are also, independently, two of the three
least-covered figures on the English side. Missing translation coverage
here tracks obscurity, not a random gap or a cutoff by time period.

2. In the first block, restrict `text_en` and `text_fr` to the 73 shared
authors, sort both by `doc_id`, and confirm the two tables now cover the
exact same set of authors. In the second block, run each language's text
through its own spaCy pipeline with `DSText.process`, and print the
resulting shapes.

In [ ]:
docs_en = (
    text_en
    .join(text_fr.select(c.doc_id), on="doc_id")
    .sort(c.doc_id)
)
docs_fr = text_fr.sort(c.doc_id)

print(docs_en.select(c.doc_id).equals(docs_fr.select(c.doc_id)))

In [ ]:
anno_en = DSText.process(docs_en, nlp_en)
anno_fr = DSText.process(docs_fr, nlp_fr)

print(anno_en.shape)
print(anno_fr.shape)

3. For each language, compute the average number of alphabetic tokens per
document and the average number of distinct lemmas per document. In the
first block, do this for English (`anno_en`); in the second, for French
(`anno_fr`).

In [ ]:
(
    anno_en
    .filter(c.is_alpha)
    .group_by(c.doc_id)
    .agg(n_tokens = pl.len(), n_lemmas = c.lemma.n_unique())
    .select(c.n_tokens.mean(), c.n_lemmas.mean())
)

In [ ]:
(
    anno_fr
    .filter(c.is_alpha)
    .group_by(c.doc_id)
    .agg(n_tokens = pl.len(), n_lemmas = c.lemma.n_unique())
    .select(c.n_tokens.mean(), c.n_lemmas.mean())
)

An English page runs about 5,184 alphabetic tokens on average, drawn from
about 1,446 distinct lemmas; a French page for the very same author runs
about 2,579 tokens from about 822 distinct lemmas. Both halve at roughly the
same rate, which matches the character-length gap between the two text
columns: French Wikipedia's coverage of these authors is real, but
consistently shorter than English Wikipedia's.

4. Compute the TF-IDF table for each language with `DSText.compute_tfidf(...,
min_df=0.01, max_df=1.0)`, the exact call the chapter itself uses for
`anno_fr`. Print the top 10 terms for `"Jane Austen"` in both languages. Do
these lists match the chapter's own description of what a top-TF-IDF list
should look like?

In [ ]:
for author in ["Jane Austen"]:
    print("EN", author, DSText.compute_tfidf(anno_en, min_df=0.01, max_df=1.0).filter(c.doc_id == author).sort(c.tfidf, descending=True).head(10).select(c.lemma).to_series().to_list())
    print("FR", author, DSText.compute_tfidf(anno_fr, min_df=0.01, max_df=1.0).filter(c.doc_id == author).sort(c.tfidf, descending=True).head(10).select(c.lemma).to_series().to_list())

No. The chapter's own text claims that TF-IDF "downweights generic terms...
and what remains are the names, places, and titles that actually identify
each author," but with `max_df=1.0` nothing actually gets downweighted
enough to matter: `the`, `of`, `and`, `in`, `to`, and `her` fill six of
Austen's ten English slots, and `de`, `le`, `à`, `et`, `un`, `être`, and `en`
fill seven of her ten French ones. This is not an artifact of the 73-author
subset — the book's own rendered table for the full 75-author English corpus
shows exactly the same pattern (`"Louis MacNeice" → "MacNeice; the; and; in;
be; of…"`, `"George Bernard Shaw" → "the; Shaw; in; and; of; be; a;…"`), so
this is the chapter's own documented behavior, not something this notebook
introduced. Without a `max_df` cutoff, a word appearing in every single
document (`the` sits at `df_docs = 73` here) still keeps an `idf` of
exactly 1, and a common English or French function word's raw in-document
count is large enough that it still outranks a genuinely rare, document-
specific proper noun.

5. Recompute the TF-IDF tables with the stricter `max_df=0.5` the chapter's
own UMAP and distance sections use elsewhere in this same chapter, for both
languages, and re-check `"Jane Austen"`, `"A. A. Milne"`, and `"T. S.
Eliot"`. Does this cutoff produce the clean lists the chapter's prose
described? Does anything still look out of place?

In [ ]:
tfidf_en = DSText.compute_tfidf(anno_en, min_df=0.01, max_df=0.5)
tfidf_fr = DSText.compute_tfidf(anno_fr, min_df=0.01, max_df=0.5)

for author in ["Jane Austen", "A. A. Milne", "T. S. Eliot"]:
    print("EN", author, tfidf_en.filter(c.doc_id == author).sort(c.tfidf, descending=True).head(10).select(c.lemma).to_series().to_list())
    print("FR", author, tfidf_fr.filter(c.doc_id == author).sort(c.tfidf, descending=True).head(10).select(c.lemma).to_series().to_list())

Mostly yes. Dropping every lemma that appears in more than half the corpus
clears out the function words from all three lists, and what is left for
Austen is her own name, her sister Cassandra, and her novels — the clean
result question 4 promised but did not deliver. Milne's and Eliot's English
lists are almost as clean, but both still contain `"parser"`, a word with no
obvious connection to either a children's author or a modernist poet.
Neither French list has anything like it. The next question tracks that
down.

6. In the first block, search both raw text columns for the substring
`"mw-parser-output"`, count how many characters match it per document,
compare the two languages, and list the documents with any matches sorted by
count. In the second block, filter `anno_en` to the token `"parser"` and
check its `upos` tag and `is_alpha` flag. In the third block, print the raw
text around the first occurrence of `"parser"` on A. A. Milne's page.

In [ ]:
print(text_en.with_columns(n = c.text.str.count_matches("mw-parser-output")).select(c.n.sum()).item(), "matches in English text")
print(text_fr.with_columns(n = c.text.str.count_matches("mw-parser-output")).select(c.n.sum()).item(), "matches in French text")

(
    text_en
    .with_columns(n = c.text.str.count_matches("mw-parser-output"))
    .filter(c.n > 0)
    .select(c.doc_id, c.n)
    .sort(c.n, descending=True)
)

In [ ]:
anno_en.filter(c.token == "parser").select(c.doc_id, c.token, c.upos, c.is_alpha).unique()

In [ ]:
idx = docs_en.filter(c.doc_id == "A. A. Milne").item(0, "text").find("parser")
docs_en.filter(c.doc_id == "A. A. Milne").item(0, "text")[idx - 120 : idx + 120]

`"mw-parser-output"` is leaked CSS from a Wikipedia coordinate template,
scraped straight into the article body along with the rest of the page text
— 33 occurrences across four English pages, concentrated in Milne's and
Eliot's, and zero anywhere in the French text. Printing the raw context
around it confirms the source: a run of repeated `.mw-parser-output
.geo-...` class names from a hidden geo-coordinate box, copy-pasted into the
extracted text eight times in a single sentence on Milne's page alone. spaCy
tags the token `parser` itself as an ordinary alphabetic `NOUN`, exactly the
way it would tag a real word, so nothing about `is_alpha` catches it the way
notebook41's backslash artifact was caught. It sails straight through
`compute_tfidf` and inflates two authors' top-term lists with a scraping
artifact that has nothing to do with either of them.

7. Reproduce the chapter's part-of-speech comparison — stack `anno_en` and
`anno_fr` with a `lang` column, group by `lang` and `upos`, and compute the
proportion of each part of speech within each language — but now on the
matched 73-author subset both tables share. Report the determiner (`DET`)
proportion for each language.

In [ ]:
(
    pl.concat([
        anno_en.with_columns(lang = pl.lit("en")),
        anno_fr.with_columns(lang = pl.lit("fr"))
    ])
    .group_by([c.lang, c.upos])
    .agg(count = pl.len())
    .with_columns(prop = c.count / c.count.sum().over(c.lang))
    .filter(c.upos == "DET")
)

8.9% of English tokens are determiners against 12.2% of French ones, the
same gap the chapter describes, and this time it is not confounded by the
population mismatch built into the chapter's own version of this
comparison: the chapter stacks all 75 English pages against only 73 French
ones, quietly comparing two different populations, while `anno_en` and
`anno_fr` here cover the identical 73 authors on both sides.

8. Compute the pairwise angle distance for each language separately with
`DSText.compute_distances(..., min_df=0.01, max_df=0.5)`. For each language,
stack the result with a copy of itself with the ID columns swapped, and find
every author's single nearest neighbor.

In [ ]:
dist_en = DSText.compute_distances(anno_en, min_df=0.01, max_df=0.5)
dist_fr = DSText.compute_distances(anno_fr, min_df=0.01, max_df=0.5)

nn_en = (
    pl.concat([
        dist_en,
        dist_en.select(doc_id=c.doc_id2, doc_id2=c.doc_id, distance=c.distance)
    ])
    .sort(c.doc_id, c.distance)
    .group_by(c.doc_id, maintain_order=True)
    .head(1)
    .rename({"doc_id2": "nn_en"})
    .drop("distance")
)

nn_fr = (
    pl.concat([
        dist_fr,
        dist_fr.select(doc_id=c.doc_id2, doc_id2=c.doc_id, distance=c.distance)
    ])
    .sort(c.doc_id, c.distance)
    .group_by(c.doc_id, maintain_order=True)
    .head(1)
    .rename({"doc_id2": "nn_fr"})
    .drop("distance")
)

nn_compare = nn_en.join(nn_fr, on="doc_id")
nn_compare

9. What proportion of authors have the exact same nearest neighbor under
both languages? Compare this to the chance rate of matching by accident
among 72 other authors.

In [ ]:
agree_rate = nn_compare.select((c.nn_en == c.nn_fr).mean()).item()
print(round(agree_rate, 3), "agreement rate")
print(round(1 / 72, 3), "chance rate")

Almost 48% of authors (35 of 73) have the identical nearest neighbor whether
the distance is computed from their English page or their French one, over
thirty times the roughly 1.4% we would expect from two independent random
guesses among 72 other authors. The two languages are far from producing
identical geometry, but the structure that TF-IDF and the angle distance
recover is not arbitrary either: it survives translation for close to half
the corpus.

10. Look at one agreement and one disagreement from `nn_compare` by eye. Ann
Radcliffe's nearest neighbor is Jane Austen in both languages; A. A. Milne's
is T. S. Eliot in English but George Orwell in French. In the first block,
print the agreement pair's top 5 TF-IDF terms in both languages for Ann
Radcliffe and Jane Austen. In the second block, print the disagreement
group's top 5 terms for A. A. Milne, T. S. Eliot, and George Orwell. In the
third block, check whether question 6's `parser` artifact is actually what is
pulling Milne toward Eliot by recomputing the English distance between them
with `parser` removed from `anno_en`.

In [ ]:
for author in ["Ann Radcliffe", "Jane Austen"]:
    print("EN", author, tfidf_en.filter(c.doc_id == author).sort(c.tfidf, descending=True).head(5).select(c.lemma).to_series().to_list())
    print("FR", author, tfidf_fr.filter(c.doc_id == author).sort(c.tfidf, descending=True).head(5).select(c.lemma).to_series().to_list())

In [ ]:
for author in ["A. A. Milne", "T. S. Eliot", "George Orwell"]:
    print("EN", author, tfidf_en.filter(c.doc_id == author).sort(c.tfidf, descending=True).head(5).select(c.lemma).to_series().to_list())
    print("FR", author, tfidf_fr.filter(c.doc_id == author).sort(c.tfidf, descending=True).head(5).select(c.lemma).to_series().to_list())

In [ ]:
milne_eliot = dist_en.filter(c.doc_id == "A. A. Milne", c.doc_id2 == "T. S. Eliot").item(0, "distance")

dist_en_no_parser = DSText.compute_distances(anno_en.filter(c.lemma != "parser"), min_df=0.01, max_df=0.5)
milne_eliot_no_parser = dist_en_no_parser.filter(c.doc_id == "A. A. Milne", c.doc_id2 == "T. S. Eliot").item(0, "distance")

print(round(milne_eliot, 3), "with parser")
print(round(milne_eliot_no_parser, 3), "without parser")

Radcliffe and Austen both write Gothic-adjacent fiction, and Radcliffe's own
top terms (`Udolpho` in English, `gothique` in French) confirm the genre
connection directly, so the agreement is easy to believe in both languages.
Milne's disagreement is harder to read off the term lists alone — neither
Eliot's nor Orwell's top terms share anything obvious with Milne's beyond
ordinary biographical scaffolding. The `parser` artifact turns out not to be
the explanation either: dropping it from `anno_en` entirely and
recomputing barely moves the Milne–Eliot distance (1.528 to 1.548) and does
not change who Milne's nearest neighbor is. Whatever pulls Milne toward
Eliot in English and toward Orwell in French
runs through the geometry of the full term-weighted vector, not through any
single term visible in a short printed list — a caution worth carrying
forward: a top-5 or top-10 list is a convenient summary for a human reader,
but it is not the same object the distance calculation actually compares.

11. Build a 2D UMAP projection for each language with `umap_text(...,
min_df=0.01, max_df=0.5, random_state=42)`, join each author's short name
and `era` from `meta`, and plot both, colored by `era`. In the first block,
do this for English (`anno_en`); in the second, for French (`anno_fr`). A
plot is an impression, not a measurement, so in the third block also compute
each projection's own same-era nearest-neighbor rate directly from the
`dr0`/`dr1` coordinates, the same check the last chapter's notebook ran on
`label`.

In [ ]:
umap_en = (
    anno_en
    .pipe(umap_text, doc_id=c.doc_id, term_id=c.lemma, n_components=2, min_df=0.01, max_df=0.5, random_state=42)
    .predict(full=True)
    .join(meta.select(c.doc_id, c.short, c.era), on="doc_id")
)

(
    umap_en
    .pipe(ggplot, aes(c.dr0, c.dr1))
    .geom_text(aes(label=c.short, color=c.era), size=6)
)

In [ ]:
umap_fr = (
    anno_fr
    .pipe(umap_text, doc_id=c.doc_id, term_id=c.lemma, n_components=2, min_df=0.01, max_df=0.5, random_state=42)
    .predict(full=True)
    .join(meta.select(c.doc_id, c.short, c.era), on="doc_id")
)

(
    umap_fr
    .pipe(ggplot, aes(c.dr0, c.dr1))
    .geom_text(aes(label=c.short, color=c.era), size=6)
)

In [ ]:
def era_nn_rate(umap_df):
    coords = umap_df.select(c.dr0, c.dr1).to_numpy()
    dist_sq = ((coords[:, None, :] - coords[None, :, :]) ** 2).sum(axis=2)
    np.fill_diagonal(dist_sq, np.inf)
    nn_idx = dist_sq.argmin(axis=1)
    eras = umap_df.select(c.era).to_series().to_numpy()
    return (eras == eras[nn_idx]).mean()

print(round(era_nn_rate(umap_en), 3), "EN same-era nearest-neighbor rate")
print(round(era_nn_rate(umap_fr), 3), "FR same-era nearest-neighbor rate")

12. With seven `era` categories of uneven size, matching by chance alone
runs around 17%. Both projections clear that bar but not by much — 20.5% for
English and 26.0% for French — confirming what the plots suggest by eye: a
weak, genuine pull toward grouping by era, not the tight separation
notebook42's own same-label check found for AG News's four disjoint topic
categories, and not nothing either. Looking back across this notebook: what
carried over unchanged between English and French, and what genuinely
differed?

The machinery carried over without a single change: the same
`DSText.process`, `compute_tfidf`, `compute_distances`, and `umap_text`
calls ran on French tokens exactly as they did on English ones, because none
of that code has any language-specific logic built in. What differed was
the input each pipeline was handed and what came out the other end. Two
authors have no French page at all, and coverage tracks obscurity rather
than era. French pages run about half the length and half the vocabulary of
their English counterparts on the very same subject. French uses
determiners more often, a grammatical fact with nothing to do with editorial
choice. And the two languages' own document geometries agree on a given
author's closest match less than half the time, even though that rate is
far above chance. None of this is a flaw in the pipeline; it is exactly the
kind of genuine cross-lingual difference the chapter's own closing paragraph
says this machinery makes possible to study systematically rather than
assert from a hunch.